# Math RDT 30M: Colab End-to-End Training and Inference

This notebook starts from existing JSONL files on Google Drive and runs the full text-only Math RDT workflow:

1. install dependencies
2. mount Google Drive
3. read and validate JSONL rows
4. train or load a byte-level BPE tokenizer
5. encode datasets
6. define a ~30M recurrent-depth transformer
7. run a tiny overfit smoke test
8. train
9. evaluate exact-answer accuracy by loop count
10. save/load checkpoints
11. run interactive inference

Expected row format:

```json
{
  "input": "...",
  "output": {
    "cot": "... Therefore, the answer is ...",
    "answer": "..."
  }
}
```


In [ ]:
# 1. Install dependencies.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "tokenizers": "tokenizers>=0.15.0",
    "tqdm": "tqdm",
    "numpy": "numpy",
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")


In [ ]:
# 2. Mount Google Drive and configure paths/hyperparameters.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped. This is expected outside Colab.")
    print(type(exc).__name__, str(exc)[:200])

DATA_GLOB = "/content/drive/MyDrive/MATH-RDT/**/*.jsonl"

WORK_DIR = Path("/content/math_rdt_work")
CKPT_DIR = Path("/content/drive/MyDrive/MATH-RDT/checkpoints")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = CKPT_DIR / "math_rdt_30m_tokenizer.json"
CHECKPOINT_PATH = CKPT_DIR / "math_rdt_30m_last.pt"

SEED = 42
MAX_ROWS = 100_000       # Set to None for full dataset loading.
VAL_ROWS = 5_000
TEST_ROWS = 5_000

VOCAB_SIZE = 16_000
MAX_SEQ_LEN = 256

# Keeping d_model=384 and vocab=16k near 30M requires an untied LM head.
# Set TIE_EMBEDDINGS=True for a smaller tied-head variant.
D_MODEL = 384
N_QUERY_HEADS = 6
N_KV_HEADS = 2
HEAD_DIM = 64
N_PRELUDE_LAYERS = 2
N_CODA_LAYERS = 2
DENSE_FFN_HIDDEN = 1536
MOE_EXPERT_HIDDEN = 768
N_ROUTED_EXPERTS = 8
N_SHARED_EXPERTS = 1
MOE_TOP_K = 2
DROPOUT = 0.1
MAX_LOOPS = 16
TIE_EMBEDDINGS = False

TRAIN_LOOP_MIN = 2
TRAIN_LOOP_MAX = 6
EVAL_LOOP_COUNTS = [2, 4, 6, 8]
INFER_DEFAULT_LOOPS = 6

BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0
TRAIN_STEPS = 500
CHECKPOINT_EVERY = 1_000
EVAL_EVERY = 500
EVAL_MAX_EXAMPLES = 200

RUN_TINY_OVERFIT = True
TINY_OVERFIT_STEPS = 80
TINY_OVERFIT_ROWS = 32
RESET_AFTER_TINY_OVERFIT = True

ANSWER_LOSS_WEIGHT = 2.0
MOE_AUX_LOSS_COEF = 0.01
LOAD_EXISTING_TOKENIZER = True
LOAD_EXISTING_CHECKPOINT = False

print("Work dir:", WORK_DIR)
print("Checkpoint dir:", CKPT_DIR)


In [ ]:
# 3. Imports, seeding, and special tokens.
import glob
import json
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print("Compute capability:", f"{major}.{minor}")

PAD = "<|pad|>"
BOS = "<|bos|>"
EOS = "<|eos|>"
INPUT = "<|input|>"
OUTPUT = "<|output|>"
COT = "<|cot|>"
ANSWER = "<|answer|>"
LOOP = "<|loop|>"
UNK = "<|unk|>"
SPECIAL_TOKENS = [PAD, BOS, EOS, INPUT, OUTPUT, COT, ANSWER, LOOP, UNK]


In [ ]:
# 4. Read, validate, canonicalize, and split existing JSONL rows.
def canonicalize_row(raw: dict, source_path: str, line_no: int) -> dict:
    if not isinstance(raw, dict):
        raise ValueError("row is not a JSON object")
    if "input" not in raw or "output" not in raw:
        raise ValueError("row must contain 'input' and 'output'")
    if not isinstance(raw["output"], dict):
        raise ValueError("row['output'] must be an object")

    question = str(raw["input"]).strip()
    cot = str(raw["output"].get("cot", "")).strip()
    answer = str(raw["output"].get("answer", "")).strip()

    if not question:
        raise ValueError("empty input")
    if not cot:
        raise ValueError("empty output.cot")
    if not answer:
        raise ValueError("empty output.answer")

    if "Therefore, the answer is" not in cot:
        cot = cot.rstrip()
        if cot and not cot.endswith("\n"):
            cot += "\n"
        cot += f"Therefore, the answer is {answer}."

    module = raw.get("module")
    if module is None and isinstance(raw.get("metadata"), dict):
        module = raw["metadata"].get("module")
    if module is None:
        module = "unknown"

    return {
        "input": question,
        "output": {"cot": cot, "answer": answer},
        "module": str(module),
        "source_path": source_path,
        "line_no": line_no,
    }


def infer_split_from_path(path: str) -> Optional[str]:
    name = Path(path).stem.lower()
    parts = re.split(r"[^a-z0-9]+", name)
    if "train" in parts:
        return "train"
    if "val" in parts or "valid" in parts or "validation" in parts:
        return "val"
    if "test" in parts:
        return "test"
    return None


def read_jsonl(path: str, cap: Optional[int] = None) -> List[dict]:
    rows = []
    bad = 0
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                raw = json.loads(line)
                rows.append(canonicalize_row(raw, path, line_no))
            except Exception as exc:
                bad += 1
                if bad <= 3:
                    print(f"Bad row skipped: {path}:{line_no}: {exc}")
            if cap is not None and len(rows) >= cap:
                break
    if bad:
        print(f"Skipped {bad} bad rows in {path}")
    return rows


def load_existing_dataset(data_glob: str) -> Tuple[List[dict], List[dict], List[dict]]:
    paths = sorted(glob.glob(data_glob, recursive=True))
    if not paths:
        raise FileNotFoundError(
            f"No JSONL files matched DATA_GLOB={data_glob!r}. "
            "Mount Google Drive and update DATA_GLOB to your existing dataset files."
        )
    print(f"Found {len(paths)} JSONL file(s).")
    for p in paths[:10]:
        print(" -", p)
    if len(paths) > 10:
        print(" ...")

    explicit = any(infer_split_from_path(p) is not None for p in paths)
    if explicit:
        rows_by_split = {"train": [], "val": [], "test": []}
        unsplit = []
        caps = {"train": MAX_ROWS, "val": VAL_ROWS, "test": TEST_ROWS}
        for path in paths:
            split = infer_split_from_path(path)
            if split is None:
                unsplit.extend(read_jsonl(path, cap=None))
            else:
                remaining = None if caps[split] is None else max(0, caps[split] - len(rows_by_split[split]))
                if remaining != 0:
                    rows_by_split[split].extend(read_jsonl(path, cap=remaining))
        if unsplit:
            print(f"Warning: {len(unsplit)} rows came from files without split names; adding them to train.")
            rows_by_split["train"].extend(unsplit[:MAX_ROWS] if MAX_ROWS is not None else unsplit)
        train_rows = rows_by_split["train"]
        val_rows = rows_by_split["val"]
        test_rows = rows_by_split["test"]
    else:
        all_rows = []
        for path in paths:
            remaining = None if MAX_ROWS is None else max(0, MAX_ROWS - len(all_rows))
            if remaining == 0:
                break
            all_rows.extend(read_jsonl(path, cap=remaining))
        rng = random.Random(SEED)
        rng.shuffle(all_rows)
        n = len(all_rows)
        n_test = min(TEST_ROWS, max(1, int(0.05 * n))) if n >= 20 else max(1, n // 10)
        n_val = min(VAL_ROWS, max(1, int(0.05 * n))) if n >= 20 else max(1, n // 10)
        n_train = max(0, n - n_val - n_test)
        train_rows = all_rows[:n_train]
        val_rows = all_rows[n_train:n_train + n_val]
        test_rows = all_rows[n_train + n_val:]

    if not train_rows:
        raise RuntimeError("No training rows were loaded.")
    if not val_rows:
        print("Validation split is empty; taking a small validation slice from train.")
        val_rows = train_rows[-min(1000, max(1, len(train_rows)//20)):]
        train_rows = train_rows[:-len(val_rows)]
    if not test_rows:
        print("Test split is empty; copying validation rows for smoke evaluation.")
        test_rows = list(val_rows)

    print(f"Rows: train={len(train_rows):,} val={len(val_rows):,} test={len(test_rows):,}")
    print("Train module counts:", dict(Counter(r["module"] for r in train_rows).most_common(10)))
    return train_rows, val_rows, test_rows


train_rows, val_rows, test_rows = load_existing_dataset(DATA_GLOB)
print("Example row:")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:1200])


In [ ]:
# 5. Train or load byte-level BPE tokenizer.
def format_training_text(row: dict) -> str:
    return (
        f"{BOS}{INPUT}\n"
        f"{row['input']}\n"
        f"{OUTPUT}{COT}\n"
        f"{row['output']['cot']}\n"
        f"{ANSWER}\n"
        f"{row['output']['answer']}{EOS}"
    )


def make_prompt(question: str) -> str:
    return f"{BOS}{INPUT}\n{question.strip()}\n{OUTPUT}{COT}\n"


def train_tokenizer(rows: List[dict], tokenizer_path: Path) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token=UNK))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )
    tokenizer.train_from_iterator((format_training_text(row) for row in rows), trainer=trainer, length=len(rows))
    tokenizer_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer.save(str(tokenizer_path))
    return tokenizer


if LOAD_EXISTING_TOKENIZER and TOKENIZER_PATH.exists():
    print("Loading tokenizer:", TOKENIZER_PATH)
    tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
else:
    print("Training tokenizer from loaded training rows...")
    tokenizer = train_tokenizer(train_rows, TOKENIZER_PATH)
    print("Saved tokenizer:", TOKENIZER_PATH)

special_ids = {tok: tokenizer.token_to_id(tok) for tok in SPECIAL_TOKENS}
if any(token_id is None for token_id in special_ids.values()):
    print("Existing tokenizer is missing required special tokens; retraining.")
    tokenizer = train_tokenizer(train_rows, TOKENIZER_PATH)
    special_ids = {tok: tokenizer.token_to_id(tok) for tok in SPECIAL_TOKENS}
print("Vocab size:", tokenizer.get_vocab_size())
print("Special ids:", special_ids)

PAD_ID = special_ids[PAD]
EOS_ID = special_ids[EOS]
ANSWER_ID = special_ids[ANSWER]


In [ ]:
# 6. Encode rows and build DataLoaders.
def encode_row(row: dict, max_seq_len: int = MAX_SEQ_LEN) -> Optional[dict]:
    prompt_text = f"{BOS}{INPUT}\n{row['input']}\n"
    target_text = f"{OUTPUT}{COT}\n{row['output']['cot']}\n{ANSWER}\n{row['output']['answer']}{EOS}"
    prompt_ids = tokenizer.encode(prompt_text).ids
    target_ids = tokenizer.encode(target_text).ids
    full_ids = prompt_ids + target_ids
    if len(full_ids) < 3:
        return None
    if len(full_ids) > max_seq_len + 1:
        return None

    answer_index = None
    for i, token_id in enumerate(full_ids):
        if token_id == ANSWER_ID:
            answer_index = i
            break

    input_ids = full_ids[:-1]
    labels = full_ids[1:]
    weights = []
    for pos in range(len(input_ids)):
        label_idx = pos + 1
        if label_idx < len(prompt_ids):
            weights.append(0.0)
        else:
            w = 1.0
            if answer_index is not None and label_idx > answer_index:
                w = ANSWER_LOSS_WEIGHT
            weights.append(w)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "loss_weights": weights,
        "answer": row["output"]["answer"],
        "module": row.get("module", "unknown"),
        "question": row["input"],
    }


class EncodedMathDataset(Dataset):
    def __init__(self, rows: List[dict], name: str):
        self.name = name
        self.examples = []
        dropped = 0
        for row in tqdm(rows, desc=f"Encoding {name}"):
            item = encode_row(row)
            if item is None:
                dropped += 1
            else:
                self.examples.append(item)
        if not self.examples:
            raise RuntimeError(f"No examples left after encoding {name}; increase MAX_SEQ_LEN or inspect data.")
        print(f"{name}: kept={len(self.examples):,} dropped_long_or_invalid={dropped:,}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_batch(batch: List[dict]) -> dict:
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, weights, attention_mask = [], [], [], []
    for item in batch:
        n = len(item["input_ids"])
        pad = max_len - n
        input_ids.append(item["input_ids"] + [PAD_ID] * pad)
        labels.append(item["labels"] + [-100] * pad)
        weights.append(item["loss_weights"] + [0.0] * pad)
        attention_mask.append([1] * n + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "loss_weights": torch.tensor(weights, dtype=torch.float32),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "answers": [x["answer"] for x in batch],
        "modules": [x["module"] for x in batch],
        "questions": [x["question"] for x in batch],
    }


train_ds = EncodedMathDataset(train_rows, "train")
val_ds = EncodedMathDataset(val_rows, "val")
test_ds = EncodedMathDataset(test_rows, "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=0)


In [ ]:
# 7. Define the recurrent-depth transformer model.
@dataclass
class ModelConfig:
    vocab_size: int
    max_seq_len: int = MAX_SEQ_LEN
    d_model: int = D_MODEL
    n_query_heads: int = N_QUERY_HEADS
    n_kv_heads: int = N_KV_HEADS
    head_dim: int = HEAD_DIM
    n_prelude_layers: int = N_PRELUDE_LAYERS
    n_coda_layers: int = N_CODA_LAYERS
    dense_ffn_hidden: int = DENSE_FFN_HIDDEN
    moe_expert_hidden: int = MOE_EXPERT_HIDDEN
    n_routed_experts: int = N_ROUTED_EXPERTS
    n_shared_experts: int = N_SHARED_EXPERTS
    moe_top_k: int = MOE_TOP_K
    dropout: float = DROPOUT
    max_loops: int = MAX_LOOPS
    pad_token_id: int = PAD_ID
    tie_embeddings: bool = TIE_EMBEDDINGS


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model: int, hidden: int, dropout: float):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.w_gate(x)) * self.w_up(x)
        return self.w_down(self.dropout(x))


class GQAAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        assert cfg.n_query_heads % cfg.n_kv_heads == 0
        assert cfg.n_query_heads * cfg.head_dim == cfg.d_model
        self.nq = cfg.n_query_heads
        self.nkv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.q_proj = nn.Linear(cfg.d_model, self.nq * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.nkv * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.nkv * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.nq * self.head_dim, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, attention_mask=None):
        bsz, seq_len, _ = x.shape
        q = self.q_proj(x).view(bsz, seq_len, self.nq, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, seq_len, self.nkv, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, seq_len, self.nkv, self.head_dim).transpose(1, 2)
        if self.nq != self.nkv:
            repeat = self.nq // self.nkv
            k = k.repeat_interleave(repeat, dim=1)
            v = v.repeat_interleave(repeat, dim=1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool).tril()
        scores = scores.masked_fill(~causal.view(1, 1, seq_len, seq_len), torch.finfo(scores.dtype).min)
        if attention_mask is not None:
            key_mask = attention_mask[:, None, None, :].bool()
            scores = scores.masked_fill(~key_mask, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores.float(), dim=-1).to(q.dtype)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bsz, seq_len, self.nq * self.head_dim)
        return self.o_proj(out)


class DenseBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.dense_ffn_hidden, cfg.dropout)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, attention_mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), attention_mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class SparseMoE(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_experts = cfg.n_routed_experts
        self.top_k = cfg.moe_top_k
        self.gate = nn.Linear(cfg.d_model, self.n_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLU(cfg.d_model, cfg.moe_expert_hidden, cfg.dropout) for _ in range(cfg.n_routed_experts)])
        self.shared = nn.ModuleList([SwiGLU(cfg.d_model, cfg.moe_expert_hidden, cfg.dropout) for _ in range(cfg.n_shared_experts)])

    def forward(self, x):
        bsz, seq_len, dim = x.shape
        flat = x.reshape(-1, dim)
        gate_logits = self.gate(flat)
        gate_probs = F.softmax(gate_logits.float(), dim=-1).to(flat.dtype)
        top_vals, top_idx = torch.topk(gate_probs, k=self.top_k, dim=-1)
        top_vals = top_vals / top_vals.sum(dim=-1, keepdim=True).clamp_min(1e-9)
        out = torch.zeros_like(flat)
        for expert_id, expert in enumerate(self.experts):
            mask = top_idx == expert_id
            if mask.any():
                token_idx, slot_idx = mask.nonzero(as_tuple=True)
                expert_out = expert(flat[token_idx])
                out[token_idx] += expert_out * top_vals[token_idx, slot_idx].unsqueeze(-1)
        if self.shared:
            shared_out = sum(expert(flat) for expert in self.shared) / len(self.shared)
            out = out + shared_out
        with torch.no_grad():
            selected = F.one_hot(top_idx, num_classes=self.n_experts).float().sum(dim=1)
            load = selected.mean(dim=0) / max(1, self.top_k)
        importance = gate_probs.float().mean(dim=0)
        aux_loss = self.n_experts * torch.sum(importance * load.to(importance.device))
        return out.view(bsz, seq_len, dim), aux_loss


class RecurrentBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.moe = SparseMoE(cfg)
        self.loop_embed = nn.Embedding(cfg.max_loops + 1, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.a_logit = nn.Parameter(torch.tensor(1.5))
        self.b_logit = nn.Parameter(torch.tensor(-1.5))
        self.transform_scale = nn.Parameter(torch.tensor(0.5))

    def forward(self, h, e, loop_idx: int, attention_mask=None):
        loop_id = min(loop_idx, self.loop_embed.num_embeddings - 1)
        loop_signal = self.loop_embed.weight[loop_id].view(1, 1, -1)
        x = h + loop_signal
        attn_out = self.dropout(self.attn(self.norm1(x), attention_mask))
        moe_out, aux_loss = self.moe(self.norm2(h + attn_out))
        moe_out = self.dropout(moe_out)
        a = 0.98 * torch.sigmoid(self.a_logit)
        b = torch.sigmoid(self.b_logit)
        transform = attn_out + moe_out
        h_next = a * h + b * e + self.transform_scale * transform
        return h_next, aux_loss


class MathRDT(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.prelude = nn.ModuleList([DenseBlock(cfg) for _ in range(cfg.n_prelude_layers)])
        self.recurrent = RecurrentBlock(cfg)
        self.coda = nn.ModuleList([DenseBlock(cfg) for _ in range(cfg.n_coda_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, n_loops: int = 6):
        bsz, seq_len = input_ids.shape
        if seq_len > self.cfg.max_seq_len:
            raise ValueError(f"Sequence length {seq_len} exceeds max_seq_len={self.cfg.max_seq_len}")
        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.prelude:
            x = block(x, attention_mask)
        e = x
        h = e
        aux_losses = []
        for loop_idx in range(1, n_loops + 1):
            h, aux_loss = self.recurrent(h, e, loop_idx, attention_mask)
            aux_losses.append(aux_loss)
        x = h
        for block in self.coda:
            x = block(x, attention_mask)
        x = self.norm(x)
        logits = self.lm_head(x)
        aux = torch.stack(aux_losses).mean() if aux_losses else torch.zeros((), device=input_ids.device)
        return logits, aux


cfg = ModelConfig(vocab_size=tokenizer.get_vocab_size())
model = MathRDT(cfg).to(device)
param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(cfg)
print(f"Parameters: total={param_count/1e6:.2f}M trainable={trainable_count/1e6:.2f}M")


In [ ]:
# 8. Loss, optimizer, scheduler, checkpoint helpers.
def compute_weighted_lm_loss(logits, labels, loss_weights):
    vocab_size = logits.size(-1)
    active = labels.ne(-100)
    safe_labels = labels.masked_fill(~active, 0)
    ce = F.cross_entropy(logits.reshape(-1, vocab_size), safe_labels.reshape(-1), reduction="none")
    ce = ce.view_as(labels)
    weights = loss_weights.to(ce.dtype) * active.to(ce.dtype)
    denom = weights.sum().clamp_min(1.0)
    return (ce * weights).sum() / denom


def batch_to_device(batch):
    return {
        "input_ids": batch["input_ids"].to(device),
        "labels": batch["labels"].to(device),
        "loss_weights": batch["loss_weights"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
    }


def make_optimizer_and_scheduler(model, total_steps: int):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def cycle(loader):
    while True:
        for batch in loader:
            yield batch


if device.type == "cuda" and torch.cuda.is_bf16_supported():
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16 if device.type == "cuda" else torch.float32
print("AMP dtype:", AMP_DTYPE)

from contextlib import nullcontext


def autocast_context():
    if device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return nullcontext()


use_scaler = device.type == "cuda" and AMP_DTYPE == torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)


def save_checkpoint(path: Path, model, optimizer, scheduler, step: int, metrics: Optional[dict] = None):
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_state": model.state_dict(),
        "model_config": asdict(model.cfg),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "step": step,
        "metrics": metrics or {},
        "tokenizer_path": str(TOKENIZER_PATH),
        "special_tokens": SPECIAL_TOKENS,
    }
    torch.save(payload, path)
    print("Saved checkpoint:", path)


def load_checkpoint(path: Path, map_location=None):
    payload = torch.load(path, map_location=map_location or device)
    loaded_cfg = ModelConfig(**payload["model_config"])
    loaded_model = MathRDT(loaded_cfg).to(device)
    loaded_model.load_state_dict(payload["model_state"])
    return loaded_model, payload


In [ ]:
# 9. Training utilities and optional tiny overfit smoke test.
def train_steps(model, loader, steps: int, *, desc: str, checkpoint_path: Optional[Path] = None, start_step: int = 0):
    optimizer, scheduler = make_optimizer_and_scheduler(model, total_steps=max(1, steps))
    iterator = cycle(loader)
    model.train()
    running = []
    progress = tqdm(range(1, steps + 1), desc=desc)
    for local_step in progress:
        optimizer.zero_grad(set_to_none=True)
        total_loss_value = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            raw_batch = next(iterator)
            batch = batch_to_device(raw_batch)
            n_loops = random.randint(TRAIN_LOOP_MIN, TRAIN_LOOP_MAX)
            with autocast_context():
                logits, aux_loss = model(batch["input_ids"], batch["attention_mask"], n_loops=n_loops)
                lm_loss = compute_weighted_lm_loss(logits, batch["labels"], batch["loss_weights"])
                loss = lm_loss + MOE_AUX_LOSS_COEF * aux_loss
                loss = loss / GRAD_ACCUM_STEPS
            if use_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            total_loss_value += float(loss.detach().cpu())
        if use_scaler:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        scheduler.step()

        global_step = start_step + local_step
        running.append(total_loss_value)
        if len(running) > 50:
            running.pop(0)
        progress.set_postfix(loss=f"{np.mean(running):.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

        if checkpoint_path is not None and CHECKPOINT_EVERY and global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(checkpoint_path, model, optimizer, scheduler, global_step, {"train_loss": float(np.mean(running))})
    return optimizer, scheduler


if RUN_TINY_OVERFIT:
    print(f"Running tiny overfit smoke test on {TINY_OVERFIT_ROWS} examples for {TINY_OVERFIT_STEPS} steps...")
    tiny_ds = torch.utils.data.Subset(train_ds, list(range(min(TINY_OVERFIT_ROWS, len(train_ds)))))
    tiny_loader = DataLoader(tiny_ds, batch_size=min(8, BATCH_SIZE), shuffle=True, collate_fn=collate_batch, num_workers=0)
    tiny_model = MathRDT(cfg).to(device)
    train_steps(tiny_model, tiny_loader, TINY_OVERFIT_STEPS, desc="tiny-overfit")
    if not RESET_AFTER_TINY_OVERFIT:
        model = tiny_model
    del tiny_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Tiny overfit smoke test finished.")


In [ ]:
# 10. Train or load the main model.
if LOAD_EXISTING_CHECKPOINT and CHECKPOINT_PATH.exists():
    print("Loading checkpoint:", CHECKPOINT_PATH)
    model, payload = load_checkpoint(CHECKPOINT_PATH)
    print("Loaded step:", payload.get("step"))
else:
    print(f"Training for {TRAIN_STEPS} optimizer steps...")
    optimizer, scheduler = train_steps(model, train_loader, TRAIN_STEPS, desc="train", checkpoint_path=CHECKPOINT_PATH)
    save_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, TRAIN_STEPS, {"train_steps": TRAIN_STEPS})


In [ ]:
# 11. Generation, answer parsing, and exact-match evaluation.
def decode_ids(ids: List[int]) -> str:
    return tokenizer.decode([int(x) for x in ids], skip_special_tokens=False)


def normalize_answer(text: str) -> str:
    text = str(text).strip()
    text = text.replace(EOS, "").strip()
    text = re.sub(r"\s+", " ", text)
    if text.endswith("."):
        text = text[:-1].strip()
    return text


def parse_final_answer(generated_text: str) -> str:
    if ANSWER in generated_text:
        tail = generated_text.split(ANSWER, 1)[1]
        tail = tail.split(EOS, 1)[0]
        lines = [line.strip() for line in tail.splitlines() if line.strip()]
        if lines:
            return normalize_answer(lines[0])
        return normalize_answer(tail)
    m = re.search(r"Therefore,\s*the answer is\s*(.+?)(?:\n|<\|eos\|>|$)", generated_text, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return normalize_answer(m.group(1))
    lines = [line.strip() for line in generated_text.splitlines() if line.strip()]
    return normalize_answer(lines[-1] if lines else "")


@torch.no_grad()
def generate_text(model, question: str, loops: int = INFER_DEFAULT_LOOPS, max_new_tokens: int = 128, temperature: float = 0.0) -> str:
    model.eval()
    prompt = make_prompt(question)
    ids = tokenizer.encode(prompt).ids
    if len(ids) >= MAX_SEQ_LEN:
        ids = ids[-(MAX_SEQ_LEN - 1):]
    for _ in range(max_new_tokens):
        input_ids = torch.tensor([ids[-MAX_SEQ_LEN:]], dtype=torch.long, device=device)
        attention_mask = torch.ones_like(input_ids)
        logits, _ = model(input_ids, attention_mask, n_loops=loops)
        next_logits = logits[0, -1]
        if temperature and temperature > 0:
            probs = F.softmax(next_logits / temperature, dim=-1)
            next_id = int(torch.multinomial(probs, num_samples=1).item())
        else:
            next_id = int(torch.argmax(next_logits).item())
        ids.append(next_id)
        if next_id == EOS_ID:
            break
        if len(ids) >= MAX_SEQ_LEN:
            break
    return decode_ids(ids)


def answer(question: str, loops: int = INFER_DEFAULT_LOOPS, max_new_tokens: int = 128, temperature: float = 0.0) -> dict:
    raw = generate_text(model, question, loops=loops, max_new_tokens=max_new_tokens, temperature=temperature)
    parsed = parse_final_answer(raw)
    cot_text = raw.split(COT, 1)[1] if COT in raw else raw
    if ANSWER in cot_text:
        cot_text = cot_text.split(ANSWER, 1)[0]
    return {
        "question": question,
        "cot": cot_text.strip(),
        "answer": parsed,
        "raw": raw,
        "loops": loops,
    }


def evaluate_generation(model, rows: List[dict], loops: int, max_examples: int = EVAL_MAX_EXAMPLES) -> dict:
    sample = rows[:max_examples]
    total = 0
    correct = 0
    by_module = defaultdict(lambda: [0, 0])
    failures = []
    for row in tqdm(sample, desc=f"eval loops={loops}"):
        result = answer(row["input"], loops=loops, max_new_tokens=128)
        pred = normalize_answer(result["answer"])
        target = normalize_answer(row["output"]["answer"])
        ok = pred == target
        total += 1
        correct += int(ok)
        mod = row.get("module", "unknown")
        by_module[mod][0] += int(ok)
        by_module[mod][1] += 1
        if not ok and len(failures) < 20:
            failures.append({
                "input": row["input"],
                "target": target,
                "pred": pred,
                "generated_cot": result["cot"],
                "raw": result["raw"],
                "module": mod,
            })
    module_acc = {m: c / n for m, (c, n) in by_module.items() if n}
    return {
        "loops": loops,
        "total": total,
        "correct": correct,
        "accuracy": correct / max(1, total),
        "module_accuracy": module_acc,
        "failures": failures,
    }


all_eval_results = []
for loops in EVAL_LOOP_COUNTS:
    metrics = evaluate_generation(model, val_rows, loops=loops, max_examples=EVAL_MAX_EXAMPLES)
    all_eval_results.append(metrics)
    print(f"loops={loops}: accuracy={metrics['accuracy']:.4f} ({metrics['correct']}/{metrics['total']})")
    print("per-module:", metrics["module_accuracy"])

print("Failure examples from last loop budget:")
print(json.dumps(all_eval_results[-1]["failures"][:3], ensure_ascii=False, indent=2)[:4000])


In [ ]:
# 12. Reload checkpoint and run interactive inference.
if CHECKPOINT_PATH.exists():
    loaded_model, loaded_payload = load_checkpoint(CHECKPOINT_PATH)
    model = loaded_model
    model.eval()
    print("Reloaded checkpoint step:", loaded_payload.get("step"))
else:
    print("No checkpoint found at", CHECKPOINT_PATH)

demo_question = val_rows[0]["input"] if val_rows else "Calculate 483 + 129."
result = answer(demo_question, loops=INFER_DEFAULT_LOOPS, max_new_tokens=128)
print(json.dumps(result, ensure_ascii=False, indent=2))
